In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.cnn_lstm_v2 import CNNLSTMV2
from src.XAI.rise import RISE
from src.rwf2000 import RWF2000Dataset
from src.config import DATASET_ROOT
from src.config import CHECKPOINT_DIR
from src.XAI.plotting_functions import plot_gradcam_over_rgb, plot_single_frame_gradcam
from scripts.common.get_device import get_available_device

experiment_root = CHECKPOINT_DIR / "CNN_LSTM_V3" / "grid_search_1" / "baseline"
device = get_available_device()

with open(experiment_root / "config.json", "r") as f:
    hyperparameters = json.load(f)

model = CNNLSTMV2(
    hidden_channels=hyperparameters["hidden_channels"],
    reduced_channels=hyperparameters["reduced_channels"],
    classifier_hidden_size=hyperparameters["classifier_hidden_size"],
    dropout=hyperparameters["dropout"],
    cnn_cutoff=hyperparameters["cnn_cutoff"],
    cnn_unfreeze_from=hyperparameters["cnn_unfreeze_from"]
).to(device)

checkpoint = torch.load(experiment_root / "best_model.pt", map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

dataset = RWF2000Dataset(
    DATASET_ROOT, 
    num_frames=hyperparameters["num_frames"], 
    input_mode=hyperparameters["input_mode"], 
    augment=False, 
    split="val", 
    return_rgb_frames=True
)

video_num = 300
video, label, rgb_frames = dataset[video_num]
video_batch = video.unsqueeze(0).to(device)
# [1, 32, 3, 224, 224]

label = label.item()

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
rise = RISE(model)
_, heatmap, logits, target_class = rise.generate_heatmap(video_batch)
prediction = logits.argmax(dim=1).item()

In [ ]:
frame_index = 0  # Change this index to visualize different frames
plot_single_frame_gradcam(heatmap, rgb_frames,frame_index, label, prediction, alpha=0.45)

processed 8/1000 masks
processed 16/1000 masks
processed 24/1000 masks
processed 32/1000 masks
processed 40/1000 masks
processed 48/1000 masks
processed 56/1000 masks
processed 64/1000 masks
processed 72/1000 masks
processed 80/1000 masks
processed 88/1000 masks
processed 96/1000 masks
processed 104/1000 masks
processed 112/1000 masks
processed 120/1000 masks
processed 128/1000 masks
processed 136/1000 masks
processed 144/1000 masks
processed 152/1000 masks
processed 160/1000 masks
processed 168/1000 masks
processed 176/1000 masks
processed 184/1000 masks
processed 192/1000 masks
processed 200/1000 masks
processed 208/1000 masks
processed 216/1000 masks
processed 224/1000 masks
processed 232/1000 masks
processed 240/1000 masks
processed 248/1000 masks
processed 256/1000 masks
processed 264/1000 masks
processed 272/1000 masks
processed 280/1000 masks
processed 288/1000 masks
processed 296/1000 masks
processed 304/1000 masks
processed 312/1000 masks
processed 320/1000 masks
processed 328

In [5]:
print(heatmap.shape)

torch.Size([32, 224, 224])
